# AutoGen 기본 샘플

이 코드 샘플에서는 [AutoGen](https://aka.ms/ai-agents/autogen) AI 프레임워크를 사용하여 기본 에이전트를 생성합니다.

이 샘플의 목표는 나중에 다양한 에이전트 패턴을 구현할 때 추가 코드 샘플에서 사용할 단계를 보여주는 것입니다.

## 필요한 Python 패키지 가져오기

In [ ]:
import os
from dotenv import load_dotenv

from autogen_agentchat.agents import AssistantAgent
from autogen_core.models import UserMessage
from autogen_ext.models.azure import AzureAIChatCompletionClient
from azure.core.credentials import AzureKeyCredential
from autogen_core import CancellationToken

from autogen_agentchat.messages import TextMessage
from autogen_agentchat.ui import Console


## 클라이언트 생성하기

이 샘플에서는 LLM에 접근하기 위해 [GitHub Models](https://aka.ms/ai-agents-beginners/github-models)를 사용합니다.

`model`은 `gpt-4o-mini`로 정의되어 있습니다. GitHub Models 마켓플레이스에서 사용 가능한 다른 모델로 변경하여 다양한 결과를 확인해 보세요.

간단한 테스트로 `What is the capital of France`라는 간단한 프롬프트를 실행합니다.

In [ ]:
load_dotenv()
client = AzureAIChatCompletionClient(
    model="gpt-4o-mini",
    endpoint="https://models.inference.ai.azure.com",
    # To authenticate with the model you will need to generate a personal access token (PAT) in your GitHub settings.
    # Create your PAT token by following instructions here: https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens
    credential=AzureKeyCredential(os.getenv("GITHUB_TOKEN")),
    model_info={
        "json_output": True,
        "function_calling": True,
        "vision": True,
        "family": "unknown",
    },
)

result = await client.create([UserMessage(content="What is the capital of France?", source="user")])
print(result)

## 에이전트 정의하기

이제 `client`를 설정하고 제대로 작동하는지 확인했으므로, `AssistantAgent`를 생성해봅시다. 각 에이전트에는 다음을 할당할 수 있습니다:
**name** - 멀티 에이전트 흐름에서 참조할 때 유용한 짧은 이름입니다.
**model_client** - 이전 단계에서 생성한 클라이언트입니다.
**tools** - 에이전트가 작업을 완료하기 위해 사용할 수 있는 사용 가능한 도구입니다.
**system_message** - LLM의 작업, 동작 및 톤을 정의하는 메타 프롬프트입니다.

시스템 메시지를 변경하여 LLM이 어떻게 응답하는지 확인할 수 있습니다. `tools`는 Lesson #4에서 다룰 예정입니다.

In [ ]:
agent = AssistantAgent(
    name="assistant",
    model_client=client,
    tools=[],
    system_message="You are a travel agent that plans great vacations",
)

## 에이전트 실행하기

아래 함수는 에이전트를 실행합니다. `on_message` 메서드를 사용하여 새 메시지로 에이전트의 상태를 업데이트합니다.

이 경우, 사용자로부터 온 새 메시지인 `"Plan me a great sunny vacation"`으로 상태를 업데이트합니다.

메시지 내용을 변경하여 LLM이 어떻게 다르게 응답하는지 확인할 수 있습니다.

In [ ]:
from IPython.display import display, HTML


async def assistant_run():
    # Define the query
    user_query = "Plan me a great sunny vacation"

    # Start building HTML output
    html_output = "<div style='margin-bottom:10px'>"
    html_output += "<div style='font-weight:bold'>User:</div>"
    html_output += f"<div style='margin-left:20px'>{user_query}</div>"
    html_output += "</div>"

    # Execute the agent response
    response = await agent.on_messages(
        [TextMessage(content=user_query, source="user")],
        cancellation_token=CancellationToken(),
    )

    # Add agent response to HTML
    html_output += "<div style='margin-bottom:20px'>"
    html_output += "<div style='font-weight:bold'>Assistant:</div>"
    html_output += f"<div style='margin-left:20px; white-space:pre-wrap'>{response.chat_message.content}</div>"
    html_output += "</div>"

    # Display formatted HTML
    display(HTML(html_output))

# Run the function
await assistant_run()